# KDS 구조기준 RAG 시스템

**건축공학 도메인 실습** — KDS 구조기준 문서를 RAG로 검색하고 Claude에게 질문합니다.

## 목표
- v1: 기본 RAG (문서 로드 → 청킹 → 임베딩 → 벡터 검색)
- v2: 하이브리드 검색 (벡터 + BM25로 조항 번호 정확도 향상)
- v3: Multi-Index RAG (RRF 통합 + Claude 답변 생성)

## 도전 과제
- 조항 번호 기반 검색 ("KDS 41 10 20")
- 한국어 전문 용어 처리
- 답변에 출처 (조항 번호) 표시

In [ ]:
# ── Setup ──────────────────────────────────────────────
import math
import re
from collections import Counter

import anthropic
import chromadb
import numpy as np
import voyageai
from dotenv import load_dotenv

load_dotenv()

claude_client = anthropic.Anthropic()
voyage_client = voyageai.Client()
MODEL = "claude-haiku-4-5"

## KDS 구조기준 샘플 데이터

실제 KDS 41 구조기준의 주요 조항을 샘플로 준비합니다.  
각 조항에 **조항 번호**를 포함하여 조항 기반 검색을 테스트합니다.

In [ ]:
KDS_SECTIONS = [
    {
        "id": "KDS-41-10-15-010",
        "title": "콘크리트 설계기준강도",
        "text": "KDS 41 10 15 (010): 콘크리트의 설계기준강도 fck는 최소 21 MPa 이상이어야 한다. 경량콘크리트의 경우 15 MPa 이상으로 할 수 있다. 고강도 콘크리트는 fck 40 MPa 이상으로 정의하며, 특수 용도에 따라 fck 80 MPa까지 적용 가능하다."
    },
    {
        "id": "KDS-41-10-15-020",
        "title": "철근 재료 기준",
        "text": "KDS 41 10 15 (020): 이형철근의 항복강도 fy는 SD400 (400 MPa) 또는 SD500 (500 MPa)을 표준으로 한다. 초고강도 철근 SD600 (600 MPa)은 설계 시 별도 검증이 필요하다. 철근의 탄성계수 Es는 200,000 MPa로 한다."
    },
    {
        "id": "KDS-41-10-20-010",
        "title": "보 설계 — 휨",
        "text": "KDS 41 10 20 (010): RC 보의 최소 인장철근비는 max(0.25*sqrt(fck)/fy, 1.4/fy) 이상이어야 한다. 이는 취성파괴를 방지하기 위한 규정이다. 최대 인장철근비는 균형 철근비의 0.75배를 초과할 수 없다."
    },
    {
        "id": "KDS-41-10-20-020",
        "title": "보 설계 — 전단",
        "text": "KDS 41 10 20 (020): 전단보강 철근(스터럽)의 최대 간격은 d/2 이하로 한다. 여기서 d는 보의 유효깊이이다. 전단력이 큰 구간에서는 간격을 d/4 이하로 줄여야 한다. 최소 전단보강 철근비는 0.35*bw*s/fy 이상으로 한다."
    },
    {
        "id": "KDS-41-10-20-030",
        "title": "기둥 설계",
        "text": "KDS 41 10 20 (030): 기둥의 최소 단면치수는 300mm 이상이어야 한다. 주근의 최소 개수는 직사각형 단면 4개, 원형 단면 6개이다. 종방향 철근비는 0.01 이상 0.08 이하로 한다. 띠철근 간격은 min(16db, 48dt, 기둥 최소치수) 이하이다."
    },
    {
        "id": "KDS-41-12-00-010",
        "title": "하중 조합",
        "text": "KDS 41 12 00 (010): 극한강도설계법의 하중조합은 다음과 같다. 1.2D + 1.6L (기본 조합), 1.2D + 1.0L + 1.0E (지진 조합), 0.9D + 1.0E (부력 검토). 여기서 D는 고정하중, L은 적재하중, E는 지진하중이다."
    },
    {
        "id": "KDS-41-10-10-020",
        "title": "고정하중",
        "text": "KDS 41 10 10 (020): 고정하중은 구조물 자체의 무게와 영구 부착물의 무게를 포함한다. 보통중량 콘크리트 단위중량 24 kN/m3, 철근콘크리트 25 kN/m3, 벽돌벽 18 kN/m3, 시멘트 모르타르 마감 20 kN/m3을 표준값으로 한다."
    },
    {
        "id": "KDS-41-10-10-030",
        "title": "적재하중",
        "text": "KDS 41 10 10 (030): 적재하중은 건축물의 용도별로 적용한다. 주거용 바닥 2.0 kN/m2, 사무실 바닥 2.5 kN/m2, 상점 바닥 4.0 kN/m2, 창고 바닥 6.0 kN/m2, 주차장 바닥 4.0 kN/m2를 적용한다."
    },
]

print(f"KDS 조항 {len(KDS_SECTIONS)}개 로드 완료")
for s in KDS_SECTIONS:
    print(f"  {s['id']}: {s['title']}")

## v1: 기본 벡터 검색 RAG

In [ ]:
# VectorIndex 구축
class VectorIndex:
    def __init__(self, collection_name="kds"):
        self.voyage_client = voyageai.Client()
        self.chroma_client = chromadb.Client()
        self.collection = self.chroma_client.create_collection(
            name=collection_name, metadata={"hnsw:space": "cosine"}
        )

    def add_documents(self, documents, ids=None):
        if ids is None:
            ids = [f"doc_{i}" for i in range(len(documents))]
        result = self.voyage_client.embed(
            texts=documents, model="voyage-3", input_type="document"
        )
        self.collection.add(
            documents=documents, embeddings=result.embeddings, ids=ids
        )

    def search(self, query, top_k=3):
        qr = self.voyage_client.embed(
            texts=[query], model="voyage-3", input_type="query"
        )
        results = self.collection.query(
            query_embeddings=qr.embeddings, n_results=top_k
        )
        return [
            {"text": results["documents"][0][i],
             "score": 1 - results["distances"][0][i],
             "id": results["ids"][0][i]}
            for i in range(len(results["documents"][0]))
        ]


# 인덱싱
texts = [s["text"] for s in KDS_SECTIONS]
ids = [s["id"] for s in KDS_SECTIONS]

vector_index = VectorIndex(collection_name="kds_v1")
vector_index.add_documents(texts, ids=ids)
print("v1: 벡터 인덱스 구축 완료")

In [ ]:
# v1 검색 테스트
test_queries = [
    "RC 보의 최소 철근비 기준",
    "KDS 41 10 20",              # 조항 번호 직접 검색
    "사무실 적재하중은 몇 kN?",
    "지진 하중 조합",
]

for q in test_queries:
    results = vector_index.search(q, top_k=2)
    print(f"\n쿼리: '{q}'")
    for i, r in enumerate(results, 1):
        print(f"  {i}위 [{r['score']:.4f}] [{r['id']}] {r['text'][:60]}...")

## v2: 하이브리드 검색 (벡터 + BM25)

In [ ]:
class BM25Index:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.documents, self.doc_ids = [], []
        self.doc_lengths, self.avg_doc_length = [], 0
        self.doc_freqs, self.doc_term_freqs = Counter(), []

    def add_documents(self, documents, ids=None):
        self.documents = documents
        self.doc_ids = ids or [f"doc_{i}" for i in range(len(documents))]
        for doc in documents:
            tokens = doc.lower().split()
            self.doc_lengths.append(len(tokens))
            tf = Counter(tokens)
            self.doc_term_freqs.append(tf)
            for t in set(tokens):
                self.doc_freqs[t] += 1
        self.avg_doc_length = sum(self.doc_lengths) / len(self.doc_lengths)

    def search(self, query, top_k=5):
        tokens = query.lower().split()
        scores = []
        n = len(self.documents)
        for i in range(n):
            score = 0
            for t in tokens:
                tf = self.doc_term_freqs[i].get(t, 0)
                df = self.doc_freqs.get(t, 0)
                idf = math.log((n - df + 0.5) / (df + 0.5) + 1)
                dl = self.doc_lengths[i]
                num = tf * (self.k1 + 1)
                den = tf + self.k1 * (1 - self.b + self.b * dl / self.avg_doc_length)
                score += idf * num / den
            scores.append(score)
        top_idx = sorted(range(n), key=lambda i: scores[i], reverse=True)[:top_k]
        return [{"text": self.documents[i], "score": scores[i], "id": self.doc_ids[i]} for i in top_idx]


# BM25 인덱싱
bm25_index = BM25Index()
bm25_index.add_documents(texts, ids=ids)
print("v2: BM25 인덱스 구축 완료")

In [ ]:
# v2 비교: 조항 번호 검색
query = "KDS 41 10 20"
print(f"쿼리: '{query}'\n")

sem = vector_index.search(query, top_k=3)
print("[Semantic]")
for i, r in enumerate(sem, 1):
    print(f"  {i}위 [{r['id']}] {r['text'][:60]}...")

bm25 = bm25_index.search(query, top_k=3)
print("\n[BM25]")
for i, r in enumerate(bm25, 1):
    print(f"  {i}위 [{r['id']}] {r['text'][:60]}...")

## v3: Multi-Index RAG (RRF + Claude)

In [ ]:
class Retriever:
    def __init__(self, vector_index, bm25_index):
        self.vector_index = vector_index
        self.bm25_index = bm25_index

    def search(self, query, top_k=3):
        sem = self.vector_index.search(query, top_k=top_k * 2)
        bm25 = self.bm25_index.search(query, top_k=top_k * 2)

        rrf_scores = {}
        for rank, r in enumerate(sem, 1):
            rrf_scores[r["id"]] = rrf_scores.get(r["id"], 0) + 1 / (60 + rank)
        for rank, r in enumerate(bm25, 1):
            rrf_scores[r["id"]] = rrf_scores.get(r["id"], 0) + 1 / (60 + rank)

        all_texts = {r["id"]: r["text"] for r in sem + bm25}
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        return [
            {"id": did, "score": score, "text": all_texts.get(did, "")}
            for did, score in sorted_docs
        ][:top_k]


retriever = Retriever(vector_index, bm25_index)
print("v3: Retriever (하이브리드) 구축 완료")

In [ ]:
def kds_rag_query(question, retriever, top_k=3):
    """KDS 구조기준 RAG 질의응답 (출처 표시 포함)"""
    results = retriever.search(question, top_k=top_k)

    context_parts = []
    for r in results:
        context_parts.append(f"[{r['id']}] {r['text']}")
    context = "\n\n".join(context_parts)

    response = claude_client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=(
            "당신은 건축구조기준(KDS) 전문가입니다. "
            "다음 KDS 조항을 참고하여 질문에 답하세요. "
            "반드시 해당 조항 번호를 인용하세요. "
            "문서에 없는 내용은 답하지 마세요.\n\n"
            f"참고 조항:\n{context}"
        ),
        messages=[{"role": "user", "content": question}]
    )

    print(f"\n검색 결과:")
    for r in results:
        print(f"  [{r['id']}] (RRF: {r['score']:.4f}) {r['text'][:50]}...")

    return response.content[0].text

In [ ]:
# v3 테스트
questions = [
    "RC 보의 최소 철근비 기준을 KDS 조항과 함께 설명해주세요.",
    "지진 하중 조합에서 고정하중과 적재하중의 계수는?",
    "사무실 건물의 적재하중은 얼마인가요?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"질문: {q}")
    answer = kds_rag_query(q, retriever)
    print(f"\n답변:\n{answer}")

## 도전 과제

1. **KDS 조항 추가**: 더 많은 KDS 조항을 `KDS_SECTIONS`에 추가하고 검색 품질을 확인하세요
2. **청킹 전략 변경**: 긴 조항을 청킹하여 인덱싱하고, 정확도를 비교하세요
3. **System Prompt 개선**: Claude가 더 정확한 답변을 생성하도록 시스템 프롬프트를 개선하세요
4. **대화형 RAG**: 사용자가 후속 질문을 할 수 있는 멀티턴 RAG를 구현하세요